# Supported data

## Setup

Import the DALI TorchVision API (object-oriented as `dtv`, functional as `dtv_fn`) alongside the standard TorchVision counterparts for comparison.

In [1]:
import torch
import torchvision.transforms.v2 as tv
import torchvision.transforms.v2.functional as tv_fn
from PIL import Image
import numpy as np

# DALI TorchVision API — object-oriented
import nvidia.dali.experimental.torchvision as dtv

# DALI TorchVision API — functional
import nvidia.dali.experimental.torchvision.v2.functional as dtv_fn

## Input types

The TorchVision API currently accepts only `PIL.Image` and `torch.Tensor` as transformation input.
The data type determines the layout: `HWC` for `PIL.Image` and `CHW` for `torch.Tensor`.

Internally, DALI converts the input to DALI tensors, but the output has the same type as the input:
a `torch.Tensor` input produces a `torch.Tensor` output, and a `PIL.Image` input produces a `PIL.Image` output.

The input data may reside in either CPU or GPU memory. It is copied to the device on which the
transformation runs. For example, if the input data is on the CPU but the transformations were
instantiated to run on the GPU, the data is copied to the GPU before processing:

In [2]:
# A CHW tensor in CPU memory
input_cpu = torch.zeros((3, 64, 64), dtype=torch.uint8)

if torch.cuda.is_available():
    # The operation runs on the GPU, but the input tensor is in CPU memory,
    # so it is copied to the GPU first (a warning about the copy is emitted).
    out = dtv_fn.resize(input_cpu, size=[32, 32], device="gpu")
    print("Input device :", input_cpu.device)
    print("Output device:", out.device)
else:
    print("CUDA not available — skipping device copy example.")

Input device : cpu
Output device: cuda:0


<div class="alert alert-info">

  Note<br>

  Copying data between devices introduces a performance penalty, so a warning is emitted whenever it happens.
  This warning is currently emitted only by the functional API. It is not yet emitted
  in the object-oriented mode and will be added in future versions.
</div>

For the best performance:

1. Instatntiate Compose with batch_size > 1 and use batched `torch.Tensor` input.
2. Keep your data in device memory to avoid copies.
3. Consider which device will consume the processed data and align the pipeline device accordingly.

## Input layouts

`Compose` builds a DALI pipeline on the first call. The input layout and type are fixed from
then on, so use a separate `Compose` instance for inputs with a different layout or type.

A `torch.Tensor` is `(..., C, H, W)`; the leading dimensions are flattened and restored in the
output. `F` is a sequence of frames (e.g. video) and `N` is a batch of samples.

| Layout | Input | Random parameters | How to get it |
|---|---|---|---|
| `HWC` | `PIL.Image` | one per image | `PIL.Image` input |
| `CHW` | 3-D tensor | one per image | default for a 3-D tensor |
| `FCHW` | 4-D or higher tensor `(..., F, C, H, W)` | one for the whole tensor, as in TorchVision | default for a tensor with at least 4 dimensions |
| `NCHW` | 4-D or higher tensor `(..., N, C, H, W)` | one per `CHW` image | extension: `batch_size` (4-D tensor), or `layout="NCHW"` with `batch_size` |
| `NFCHW` | 5-D or higher tensor `(..., N, F, C, H, W)` | one per `FCHW` sequence | extension: `batch_size` with a 5-D tensor, or `layout="NFCHW"` with `batch_size` |

`NHWC`, `FHWC` and `NFHWC` are not supported yet.